# CAI 5607 — Homework 2
## Q5. Build and Adapt a Tiny Transformer
**45 points: A 6 · B 12 · C 7 · D 10 · E 10**

**Workflow:** text → shifted sequences → causal Transformer → pre-training → response-only SFT → held-out comparison.

Use the **new `input.txt` in this package**, not Homework 1's corpus. It contains original,
AI-authored fictional campus narratives, not real student records. The issue-tagging data
are separate synthetic teaching examples, not a diagnostic benchmark.

Keep this notebook, `input.txt`, `hw2_utils.py`, and `data/` in the same directory.
Complete every code and written-response `TODO`, and retain the checks. The starter is
intentionally incomplete; the **completed** notebook must run from beginning to end.

Choose settings using training/validation only. Run the final test after those choices
are fixed, and do not tune based on test results. No minimum accuracy or perplexity is
required. No external LLM, pretrained weights, or paid API is needed for Q5.

**Setup:** install `requirements.txt`; see `README.md`.

In [ ]:
from pathlib import Path
import json
from collections import Counter
import math
import platform

import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from hw2_utils import (
    CharCodec, LABELS, IGNORE_INDEX, seed_everything, load_records, assert_split_disjoint,
    text_batch, sequence_cross_entropy, fit_bigram, evaluate_bigram,
    evaluate_text, train_steps, cpu_state, generate_text, evaluate_task,
    evaluate_sft_loss, save_json, save_jsonl,
)

SEED = 42
CONTEXT_LENGTH = 64
D_MODEL = 64
N_HEADS = 4
N_LAYERS = 2
D_FF = 256
BATCH_SIZE = 32
PRETRAIN_STEPS = 1500
SFT_STEPS = 400
PRETRAIN_LR = 1e-3
SFT_LR = 3e-4
EVAL_INTERVAL = 100

seed_everything(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cpu":
    torch.set_num_threads(2)  # Avoid excessive thread overhead for this tiny network.
OUT = Path("outputs")
OUT.mkdir(exist_ok=True)
print("Device:", device, "| Python:", platform.python_version(), "| PyTorch:", torch.__version__)

required_paths = [Path("input.txt"), Path("hw2_utils.py")] + [
    Path("data") / name for name in ("sft_train.jsonl", "sft_val.jsonl", "sft_test.jsonl")
]
missing = [str(p) for p in required_paths if not p.is_file()]
if missing:
    raise FileNotFoundError("Extract the complete starter ZIP here. Missing: " + ", ".join(missing))


## A. Prepare the data — 6 points
Use the **new campus corpus** with an **80/10/10 contiguous split** and context length 64.
`CharCodec` builds an inventory of ordinary characters and adds separately indexed
BOS, EOS, and PAD tokens. Keep this vocabulary unchanged through pre-training and SFT.
Only this fixed character inventory is shared: training never uses held-out text windows,
counts, or responses. Every ordinary character in the supplied corpus also appears in
its training partition.

Complete `split_corpus`. Report corpus length, split lengths, ordinary-character vocabulary
size, and total vocabulary size. The provided sampler creates input and shifted-target
batches of shape `[B, 64]` **inside one split**. Display three decoded examples below.

In [ ]:
text = Path("input.txt").read_text(encoding="utf-8")
codec = CharCodec(text)
vocab_size = codec.vocab_size
assert codec.decode(codec.encode("hello")) == "hello"
data = torch.tensor(codec.encode(text), dtype=torch.long)

def split_corpus(sequence):
    # TODO: first 80% train; next 10% validation; final 10% test.
    # Use integer boundaries at 0.8*N and 0.9*N.
    raise NotImplementedError("TODO A: return train, validation, and test tensors.")

train_data, val_data, test_data = split_corpus(data)
assert len(train_data) + len(val_data) + len(test_data) == len(data)
assert torch.equal(torch.cat([train_data, val_data, test_data]), data)
assert all(len(s) > CONTEXT_LENGTH for s in (train_data, val_data, test_data))
print("Characters:", len(text), "| Character vocabulary:", len(codec.chars), "| Total vocabulary:", vocab_size)
print("Train / validation / test lengths:", len(train_data), len(val_data), len(test_data))
print("First 200 characters:", repr(text[:200]))

In [ ]:
# Provided window sampler: y is shifted once relative to x.
xb, yb = text_batch(train_data, CONTEXT_LENGTH, BATCH_SIZE, device)
assert xb.shape == yb.shape == (BATCH_SIZE, CONTEXT_LENGTH)
assert torch.equal(xb[:, 1:], yb[:, :-1])
for i in range(3):
    print("Input :", repr(codec.decode(xb[i].tolist())))
    print("Target:", repr(codec.decode(yb[i].tolist())))
print("Input and target shapes:", tuple(xb.shape), tuple(yb.shape))

## B. Implement causal multi-head self-attention — 12 points
Implement the attention math using tensor operations, `nn.Linear`, and softmax.
Do **not** use `nn.Transformer`, `nn.MultiheadAttention`, or a fused attention function
in this part; the purpose is to expose the mechanism.

Shapes: `[B,L,64] → Q,K,V each [B,4,L,16] → scores/weights [B,4,L,L] → [B,L,64]`.
Use a lower-triangular mask **including the diagonal**. Mask future scores with
negative infinity **before** row-wise softmax. The upper-triangular weights must be zero.

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model=D_MODEL, n_heads=N_HEADS, context_length=CONTEXT_LENGTH):
        super().__init__()
        if d_model % n_heads:
            raise ValueError("d_model must be divisible by n_heads")
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.proj = nn.Linear(d_model, d_model)
        allowed = torch.tril(torch.ones(context_length, context_length, dtype=torch.bool))
        self.register_buffer("allowed", allowed[None, None], persistent=False)
        self.last_attention = None

    def forward(self, x):
        B, L, D = x.shape
        if not 1 <= L <= self.allowed.shape[-1]:
            raise ValueError("Attention length exceeds the configured context window")

        # TODO 1: project and split into q, k, v.
        # TODO 2: reshape each from [B,L,D] to [B,H,L,d_k].
        # TODO 3: compute scaled dot-product scores.
        # TODO 4: apply self.allowed[:,:,:L,:L] before softmax.
        # TODO 5: compute weights and combine values.
        # TODO 6: merge heads back into [B,L,D]; assign the result to mixed.
        # Keep the provided self.proj call below: it is the output projection.
        raise NotImplementedError("TODO B: implement attention and define weights and mixed.")

        self.last_attention = weights.detach()  # For tests/visualization, not the loss.
        return self.proj(mixed)

In [ ]:
# Preserve these checks.
attention = CausalSelfAttention().to(device)
x = torch.randn(2, 7, D_MODEL, device=device, requires_grad=True)
y = attention(x)
a = attention.last_attention
assert y.shape == (2, 7, D_MODEL)
assert a.shape == (2, N_HEADS, 7, 7)
assert torch.allclose(a.sum(dim=-1), torch.ones_like(a.sum(dim=-1)), atol=1e-6)
assert torch.count_nonzero(torch.triu(a, diagonal=1)) == 0
assert torch.isfinite(y).all()
y.square().mean().backward()
assert attention.qkv.weight.grad is not None
assert torch.isfinite(attention.qkv.weight.grad).all()
assert attention.qkv.weight.grad.abs().sum() > 0
print("Attention shape, normalization, causal mask, and gradient checks passed.")
for name, block_gradient in zip(("query", "key", "value"), attention.qkv.weight.grad.chunk(3, dim=0)):
    assert block_gradient.abs().sum() > 0, f"No gradient through the {name} projection."
assert x.grad is not None and torch.isfinite(x.grad).all()

# Independent numerical check of scaling, masking, and value mixing for a tiny one-head case.
# Expected outputs are supplied as test values; implement your general computation above.
tiny_attention = CausalSelfAttention(d_model=2, n_heads=1, context_length=3).to(device)
with torch.no_grad():
    tiny_attention.qkv.weight.copy_(torch.eye(2, device=device).repeat(3, 1))
    tiny_attention.qkv.bias.zero_()
    tiny_attention.proj.weight.copy_(torch.eye(2, device=device))
    tiny_attention.proj.bias.zero_()
    tiny_input = torch.tensor([[[1., 0.], [0., 1.], [1., 1.]]], device=device)
    tiny_output = tiny_attention(tiny_input)
    expected_weights = torch.tensor(
        [[1., 0., 0.], [0.33023845, 0.66976155, 0.],
         [0.24825508, 0.24825508, 0.50348984]], device=device)
    expected_output = torch.tensor(
        [[1., 0.], [0.33023845, 0.66976155], [0.75174492, 0.75174492]], device=device)
    assert torch.allclose(tiny_attention.last_attention[0, 0], expected_weights, atol=1e-6)
    assert torch.allclose(tiny_output[0], expected_output, atol=1e-6)
print("Independent scaled-attention calculation passed.")

# Verify that changing future inputs does not change earlier outputs.
was_training = attention.training
attention.eval()

with torch.no_grad():
    original_input = x.detach()
    modified_input = original_input.clone()

    # Zero-based indexing:
    # Keep positions 0, 1, 2, 3 unchanged.
    # Modify future positions 4, 5, 6.
    cutoff = 3

    # THIS is where the future inputs change.
    modified_input[:, cutoff + 1:, :] = torch.randn_like(
        modified_input[:, cutoff + 1:, :]
    )

    # Confirm that only the future portion was modified.
    assert torch.equal(
        original_input[:, :cutoff + 1, :],
        modified_input[:, :cutoff + 1, :],
    )
    assert not torch.equal(
        original_input[:, cutoff + 1:, :],
        modified_input[:, cutoff + 1:, :],
    )

    # Run the same attention layer on both inputs.
    original_output = attention(original_input)
    modified_output = attention(modified_input)

    # Outputs through the cutoff must remain unchanged.
    assert torch.allclose(
        original_output[:, :cutoff + 1, :],
        modified_output[:, :cutoff + 1, :],
        atol=1e-6,
        rtol=1e-5,
    ), "Earlier outputs changed after modifying future inputs."

attention.train(was_training)
print("Future-input perturbation check passed.")

## C. Assemble the decoder-only Transformer — 7 points
Each block uses **pre-LayerNorm** and two residual connections:
`h = x + Attention(LN1(x))`; `output = h + FFN(LN2(h))`.
Use two blocks, learned token/position embeddings, and a final LayerNorm + vocabulary
projection. This assignment uses **no dropout** and untied input/output weights.

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.ln1 = nn.LayerNorm(D_MODEL)
        self.attn = CausalSelfAttention()
        self.ln2 = nn.LayerNorm(D_MODEL)
        self.ffn = nn.Sequential(nn.Linear(D_MODEL, D_FF), nn.GELU(), nn.Linear(D_FF, D_MODEL))

    def forward(self, x):

        # TODO: apply the two pre-LN sublayers with residual connections.
        raise NotImplementedError("TODO C: implement this forward pass.")


class TinyTransformer(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.context_length = CONTEXT_LENGTH
        self.token_embedding = nn.Embedding(vocab_size, D_MODEL)
        self.position_embedding = nn.Embedding(CONTEXT_LENGTH, D_MODEL)
        self.blocks = nn.ModuleList([TransformerBlock() for _ in range(N_LAYERS)])
        self.final_norm = nn.LayerNorm(D_MODEL)
        self.head = nn.Linear(D_MODEL, vocab_size)

    def forward(self, token_ids):
        B, L = token_ids.shape
        if not 1 <= L <= self.context_length:
            raise ValueError("Input length is outside the supported context window")

        # TODO: add token embeddings and positions 0,...,L-1; apply all blocks;
        # then return final_norm + vocabulary projection logits [B,L,V].
        # Do not flatten the sequence or apply output softmax.
        raise NotImplementedError("TODO C: implement this forward pass.")


seed_everything(SEED)
model = TinyTransformer(vocab_size).to(device)
print(model)

In [ ]:
xb, yb = text_batch(train_data, CONTEXT_LENGTH, BATCH_SIZE, device)
logits = model(xb)
assert logits.shape == (BATCH_SIZE, CONTEXT_LENGTH, vocab_size)
num_parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Input:", tuple(xb.shape), "| Logits:", tuple(logits.shape))
print("Trainable parameters:", num_parameters)

# Causality check: edit only positions strictly after cut, then compare the prefix logits.
model.eval()
original = xb[:2].clone()
changed = original.clone()
cut = 10
changed[:, cut+1:] = (changed[:, cut+1:] + 1) % len(codec.chars)
with torch.no_grad():
    before, after = model(original), model(changed)
assert torch.allclose(before[:, :cut+1], after[:, :cut+1], atol=1e-6, rtol=1e-5)
model.train()
print("Future-token edit did not change earlier logits.")
model.zero_grad(set_to_none=True)
sequence_cross_entropy(model(xb), yb).backward()
for name, parameter in model.named_parameters():
    assert parameter.grad is not None, f"Parameter disconnected from loss: {name}"
    assert torch.isfinite(parameter.grad).all(), f"Non-finite gradients: {name}"
assert model.head.weight is not model.token_embedding.weight, "Do not tie input/output weights."
print("Full-model gradients are finite and input/output weights are untied.")

**C response:** Report the trainable parameter count. Explain why the model returns one
logit vector per input position and why it must not apply output softmax before cross-entropy.

**Response:** TODO

## D. Pre-train and evaluate — 10 points
Train from random initialization on the **training** text. This is a small-scale
analogue of next-token pre-training, not a foundation model. Keep the final pre-trained
checkpoint for comparison with SFT.

The training objective supervises every shifted position in sampled windows. To make
the plotted curves comparable, both plotted NLLs use **the first 512 full-context target
positions** in their partitions. Final reported NLLs score each eligible target once,
with 64 context characters. The supplied bigram baseline scores the same target positions.

Defaults: 1,500 pre-training steps; AdamW, lr=0.001, weight decay=0.01; gradient-norm
clipping at 1.0. Use 1,000–2,000 steps, or fewer on a slow computer and document the change.

In [ ]:
# Supplied baseline. Do not reuse HW1 scores: the split, context protocol, and vocabulary changed.
bigram_probs = fit_bigram(train_data, vocab_size)
bigram_train_nll = evaluate_bigram(bigram_probs, train_data, CONTEXT_LENGTH)
bigram_val_nll = evaluate_bigram(bigram_probs, val_data, CONTEXT_LENGTH)
print(f"Bigram training NLL: {bigram_train_nll:.4f}")
print(f"Bigram validation NLL: {bigram_val_nll:.4f} | Perplexity: {math.exp(bigram_val_nll):.4f}")

# Re-initialize so rerunning earlier shape/test cells cannot change the training initialization.
seed_everything(SEED)
model = TinyTransformer(vocab_size).to(device)

def pretrain_metrics():
    return {
        "train_nll": evaluate_text(model, train_data, CONTEXT_LENGTH, device, limit=512),
        "val_nll": evaluate_text(model, val_data, CONTEXT_LENGTH, device, limit=512),
    }

pretrain_history, pretrain_seconds = train_steps(
    model,
    get_batch=lambda: text_batch(train_data, CONTEXT_LENGTH, BATCH_SIZE, device),
    steps=PRETRAIN_STEPS, lr=PRETRAIN_LR, device=device,
    evaluate=pretrain_metrics, interval=EVAL_INTERVAL,
)
base_state = cpu_state(model)
torch.save(base_state, OUT / "base_state.pt")
save_json(OUT / "pretrain_history.json", pretrain_history)
print(f"Pre-training seconds (including periodic validation): {pretrain_seconds:.2f}")

In [ ]:
plt.figure(figsize=(8, 4.5))
plt.plot([r["step"] for r in pretrain_history], [r["train_nll"] for r in pretrain_history], label="Train: fixed 512 targets")
plt.plot([r["step"] for r in pretrain_history], [r["val_nll"] for r in pretrain_history], label="Validation: fixed 512 targets")
plt.xlabel("Pre-training step")
plt.ylabel("Full-context NLL (nats/character)")
plt.title("Tiny Transformer Pre-training")
plt.legend()
plt.tight_layout()
plt.savefig(OUT / "pretraining_loss.png", dpi=160)
plt.show()

base_train_nll = evaluate_text(model, train_data, CONTEXT_LENGTH, device)
base_val_nll = evaluate_text(model, val_data, CONTEXT_LENGTH, device)
print(f"Full training NLL: {base_train_nll:.4f}")
print(f"Full validation NLL: {base_val_nll:.4f} | Perplexity: {math.exp(base_val_nll):.4f}")
# Save the ordinary-character order and separate special-token IDs with the checkpoint.
save_json(OUT / "vocabulary.json", {
    "chars": codec.chars, "bos_id": codec.bos_id, "eos_id": codec.eos_id,
    "pad_id": codec.pad_id, "vocabulary_size": vocab_size,
})


In [ ]:
# Two temperatures; the prefix is supplied, and each continuation contains 300 NEW characters.
prefix = text[:16]
text_samples = []
for temperature in (0.8, 1.2):
    seed_everything(SEED + 1)
    sample = generate_text(model, codec, prefix, 300, temperature, device)
    assert len(sample) - len(prefix) == 300
    text_samples.append({"temperature": temperature, "prefix": prefix, "text": sample})
    print(f"\nTemperature {temperature} | prefix {prefix!r}\n{sample}")
save_json(OUT / "text_samples.json", text_samples)

In [ ]:
# Plot a MEASURED attention map, not a hand-crafted example. No semantic claim is assumed.
attention_text = text[:24]  # An actual sequence from the training partition.
model.eval()
with torch.no_grad():
    model(torch.tensor([codec.encode(attention_text)], dtype=torch.long, device=device))
weights = model.blocks[0].attn.last_attention[0, 0].cpu().numpy()
labels = [{" ": "[space]", "\n": "[newline]"}.get(c, c) for c in attention_text]
plt.figure(figsize=(7, 6))
plt.imshow(weights, vmin=0, vmax=1)
plt.xticks(range(len(labels)), labels, rotation=90)
plt.yticks(range(len(labels)), labels)
plt.xlabel("Key position (read from)")
plt.ylabel("Query position (updated)")
plt.title("Measured Attention: Layer 1, Head 1")
plt.colorbar(label="Attention weight")
plt.tight_layout()
plt.savefig(OUT / "attention_map.png", dpi=160)
plt.show()

**D response:** Compare validation NLL/perplexity with the bigram. Discuss the two
samples and one pattern in the attention map. Report the training time and device. 

**Response:** TODO

## E. Fine-tune and compare — 10 points
Continue from `base_state`. Use the **160 training** task records to update parameters
and the **40 validation** records for development. Reserve **40 test** records for the final
section. Wording templates and asset identifiers are disjoint across splits. All four labels
have equal frequency within each split.

Labels: `leak`, `noise`, `heat`, and `unknown` (no specific symptom in the other three
categories). Only `record["prompt"]` is supplied as the context. Record IDs, template IDs,
asset metadata, and reference responses must not be added to the input prompt.

Construct `[BOS] + prompt + response + [EOS]`, then shift once: `inputs = full[:-1]` and
`targets = full[1:]`. Keep the prompt in the inputs. In the **targets**, mask prompt and
PAD positions with `-100`; supervise every response character **and EOS**. BOS has no
prediction target after the shift. The first response target is at
`len([BOS] + encode(prompt)) - 1`.

Right-pad inputs to 64 using `codec.pad_id`; right-pad targets with `-100`. Never put
`-100` in an embedding input. The causal mask stops valid positions from reading future
padding, while the loss mask ignores PAD targets. Do not shift again in the loss.

Fine-tune **all weights** with a **new** AdamW optimizer: default 400 steps, learning rate
`3e-4`. Use 300–600 steps, or fewer with documentation. The provided helper starts a new
optimizer on every call. Plot response-only train/validation NLL.


In [ ]:
sft_train = load_records("data/sft_train.jsonl")
sft_val = load_records("data/sft_val.jsonl")
assert_split_disjoint(sft_train, sft_val)
assert len(sft_train) == 160 and len(sft_val) == 40
print("SFT training / validation examples:", len(sft_train), len(sft_val))
print("Example:", sft_train[0])

def make_sft_example(record, codec, length=CONTEXT_LENGTH):
    # TODO: build and shift the sequence, mask all non-response targets, then right-pad.
    # Return two torch.long tensors of shape [length]: input IDs and target labels.
    raise NotImplementedError("TODO E: build, shift once, mask targets, and right-pad.")

def tensorize(records):
    pairs = [make_sft_example(r, codec) for r in records]
    return torch.stack([p[0] for p in pairs]), torch.stack([p[1] for p in pairs])

sft_x, sft_y = tensorize(sft_train)
sv_x, sv_y = tensorize(sft_val)
assert sft_x.shape == sft_y.shape == (len(sft_train), CONTEXT_LENGTH)
for record, x, y in zip(sft_train, sft_x, sft_y):
    expected = codec.encode(record["response"]) + [codec.eos_id]
    assert y[y != IGNORE_INDEX].tolist() == expected
    first = 1 + len(codec.encode(record["prompt"])) - 1
    assert y[first].item() == expected[0]
    assert (y[:first] == IGNORE_INDEX).all()
    assert (y[x == codec.pad_id] == IGNORE_INDEX).all()
print("Response-only target checks passed.")
for records, xx, yy in ((sft_train, sft_x, sft_y), (sft_val, sv_x, sv_y)):
    for record, x, y in zip(records, xx, yy):
        prompt_ids = [codec.bos_id] + codec.encode(record["prompt"])
        response_ids = codec.encode(record["response"]) + [codec.eos_id]
        full = prompt_ids + response_ids
        valid_length = len(full) - 1
        start = len(prompt_ids) - 1
        assert x.dtype == y.dtype == torch.long
        assert x[:valid_length].tolist() == full[:-1], "Do not mask or remove the prompt input."
        assert (x[valid_length:] == codec.pad_id).all()
        assert (x >= 0).all() and (x < vocab_size).all()
        expected_positions = list(range(start, start + len(response_ids)))
        actual_positions = (y != IGNORE_INDEX).nonzero().flatten().tolist()
        assert actual_positions == expected_positions, "Check the once-shifted response boundary."
        assert y[y != IGNORE_INDEX].tolist() == response_ids
print("Training and validation inputs, shifts, masks, and EOS supervision checked.")


In [ ]:
# Show target alignment for one record. -100 is a loss label, never an embedding input.
i = 0
print("Prompt:", repr(sft_train[i]["prompt"]))
print("Supervised target sequence:", codec.decode(sft_y[i][sft_y[i] != IGNORE_INDEX].tolist()))
print("Supervised positions:", (sft_y[i] != IGNORE_INDEX).nonzero().flatten().tolist())
print("position | input token  | target token | used in loss")
for position in range(CONTEXT_LENGTH):
    input_name = repr(codec.decode([sft_x[i, position].item()]))
    target = sft_y[i, position].item()
    target_name = "[ignored]" if target == IGNORE_INDEX else repr(codec.decode([target]))
    print(f"{position:8d} | {input_name:12s} | {target_name:12s} | {target != IGNORE_INDEX}")


In [ ]:
# Compare on validation while developing; reserve test for the final section below.
model.load_state_dict(base_state)
base_task_val_metrics, _ = evaluate_task(model, codec, sft_val, device)
print("Base task validation:", base_task_val_metrics)

seed_everything(SEED + 2)
sft_model = TinyTransformer(vocab_size).to(device)
sft_model.load_state_dict(base_state)

def sft_batch():
    idx = torch.randint(len(sft_x), (BATCH_SIZE,))
    return sft_x[idx].to(device), sft_y[idx].to(device)

def sft_metrics():
    return {"train_response_nll": evaluate_sft_loss(sft_model, sft_x, sft_y, device),
            "val_response_nll": evaluate_sft_loss(sft_model, sv_x, sv_y, device)}

sft_history, sft_seconds = train_steps(
    sft_model, get_batch=sft_batch, steps=SFT_STEPS, lr=SFT_LR,
    device=device, evaluate=sft_metrics, interval=EVAL_INTERVAL,
)
sft_state = cpu_state(sft_model)
torch.save(sft_state, OUT / "sft_state.pt")
save_json(OUT / "sft_history.json", sft_history)
sft_task_val_metrics, val_responses = evaluate_task(sft_model, codec, sft_val, device)
save_jsonl(OUT / "sft_validation_responses.jsonl", val_responses)
print("SFT task validation:", sft_task_val_metrics)
print(f"SFT seconds (including periodic validation): {sft_seconds:.2f}")

In [ ]:
plt.figure(figsize=(8, 4.5))
plt.plot([r["step"] for r in sft_history], [r["train_response_nll"] for r in sft_history], label="Training")
plt.plot([r["step"] for r in sft_history], [r["val_response_nll"] for r in sft_history], label="Validation")
plt.xlabel("SFT step")
plt.ylabel("Response + EOS NLL (nats/token)")
plt.title("Supervised Fine-tuning")
plt.legend()
plt.tight_layout()
plt.savefig(OUT / "sft_loss.png", dpi=160)
plt.show()

### Final test — run only after fixing settings
This section is included in the final end-to-end execution. Do not tune settings based
on its outputs. The three text models use the **same full-context test targets**; the
bigram only uses the preceding character to predict them. Task accuracy uses unrestricted
greedy generation, with EOS stopping or a maximum of 12 new tokens. Strip outer whitespace
and lowercase; extra words, malformed labels, and generated PAD/BOS are errors.

The final cell also reports a constant-label baseline chosen from the training labels.
Task accuracy and general-text NLL measure different abilities. Fine-tuning may help one
and hurt the other. Improvement is **not** required for full credit.

In [ ]:
sft_test = load_records("data/sft_test.jsonl")
assert len(sft_test) == 40
assert_split_disjoint(sft_train, sft_val, sft_test)

base_model = TinyTransformer(vocab_size).to(device)
base_model.load_state_dict(base_state)
sft_model.load_state_dict(sft_state)
base_test_nll = evaluate_text(base_model, test_data, CONTEXT_LENGTH, device)
sft_test_nll = evaluate_text(sft_model, test_data, CONTEXT_LENGTH, device)
bigram_test_nll = evaluate_bigram(bigram_probs, test_data, CONTEXT_LENGTH)
# A constant-label baseline is chosen from TRAINING labels, never test labels.
majority_label = max(LABELS, key=lambda label: sum(r["response"] == label for r in sft_train))
majority_accuracy = sum(r["response"] == majority_label for r in sft_test) / len(sft_test)
base_task_metrics, base_responses = evaluate_task(base_model, codec, sft_test, device)
sft_task_metrics, sft_responses = evaluate_task(sft_model, codec, sft_test, device)
save_jsonl(OUT / "base_test_responses.jsonl", base_responses)
save_jsonl(OUT / "sft_test_responses.jsonl", sft_responses)

results = {
    "configuration": {"seed": SEED, "context_length": CONTEXT_LENGTH, "d_model": D_MODEL,
        "heads": N_HEADS, "layers": N_LAYERS, "d_ff": D_FF, "batch_size": BATCH_SIZE,
        "pretrain_steps": PRETRAIN_STEPS, "sft_steps": SFT_STEPS,
        "pretrain_lr": PRETRAIN_LR, "sft_lr": SFT_LR, "dropout": 0.0,
        "weight_decay": 0.01, "gradient_clip_norm": 1.0},
    "environment": {"python": platform.python_version(), "torch": torch.__version__,
        "device": str(device), "cpu_threads": torch.get_num_threads()},
    "corpus_characters": len(text), "ordinary_vocabulary_size": len(codec.chars),
    "vocabulary_size": vocab_size,
    "corpus_sha256": __import__("hashlib").sha256(text.encode("utf-8")).hexdigest(),
    "split_lengths": {"train": len(train_data), "validation": len(val_data), "test": len(test_data)},
    "bigram_training_nll": bigram_train_nll, "bigram_validation_nll": bigram_val_nll,
    "bigram_validation_perplexity": math.exp(bigram_val_nll),
    "trainable_parameters": num_parameters,
    "eligible_text_test_targets": len(test_data) - CONTEXT_LENGTH,
    "pretraining_seconds_including_validation": pretrain_seconds,
    "sft_seconds_including_validation": sft_seconds,
    "base_train_nll": base_train_nll, "base_validation_nll": base_val_nll,
    "text_test": {name: {"nll": nll, "perplexity": math.exp(nll)}
        for name, nll in [("bigram", bigram_test_nll), ("base", base_test_nll), ("sft", sft_test_nll)]},
    "task_test": {"base": base_task_metrics, "sft": sft_task_metrics},
    "constant_label_baseline": {"label_chosen_from_training": majority_label, "accuracy": majority_accuracy},
}
save_json(OUT / "results.json", results)
print(json.dumps(results, indent=2))
# Save all validation responses from BOTH checkpoints as well as both final test response files.
_, base_val_responses = evaluate_task(base_model, codec, sft_val, device)
save_jsonl(OUT / "base_validation_responses.jsonl", base_val_responses)
assert len(base_responses) == len(sft_responses) == 40
print("\nTask mistakes for your error analysis (up to two per checkpoint):")
for checkpoint, rows in (("pre-trained", base_responses), ("fine-tuned", sft_responses)):
    mistakes = [r for r in rows if not r["correct"]]
    print(checkpoint, "errors:", len(mistakes))
    for row in mistakes[:2]:
        print(json.dumps(row, ensure_ascii=False))


### E response — 150–200 words
Use **your measured results** to compare the two checkpoints on task accuracy and
general-text NLL/perplexity. Discuss possible forgetting and at least two actual
errors (or all errors if fewer than two; state explicitly when there are none).
Do not assume that fine-tuning improves every metric.

**Response:** TODO

## Reproducibility and submission
Complete these fields and include `outputs/` in your submission.

- Hyperparameter changes and reasons (if any): **TODO**
- Source/code citations: **TODO**
- AI-tool disclosure, as required by course policy — tool, purpose, material used,
  and what you verified or changed: **TODO**

**Submit:** the completed `Programming_Part_Transformer.ipynb`, an executed HTML or PDF
export, additional/modified `.py` files, and `outputs/` with plots, measurements, generated 
text, and all saved task responses. Do not invent or manually replace measured outputs.

**Supplied implementation defaults:** weight decay 0.01, gradient norm clipping at 1.0,
CPU thread count 2, untied embeddings/output head, and biases enabled in linear layers.
These clarify the template; the assessed architecture and training settings follow Q5.